In [ ]:
if NeuralProphet is not None:
    save_dir = repo_root / 'models'
    save_dir.mkdir(exist_ok=True)
    
    # Save NeuralProphet model
    model_path = save_dir / 'neuralprophet_model'
    model_path.mkdir(exist_ok=True)
    
    # NeuralProphet uses torch.save internally
    import torch
    torch.save(model, model_path / 'model.pt')
    print(f'✓ Model saved: {model_path}/model.pt')
    
    # Save forecast results
    import joblib
    forecast_path = save_dir / 'neuralprophet_forecast.pkl'
    joblib.dump(forecast, forecast_path)
    print(f'✓ Forecast saved: {forecast_path}')
    
    # Save config
    config = {
        'n_forecasts': 10,
        'n_lags': 30,
        'train_size': len(df_train),
        'test_size': len(df_test),
        'mae': mae,
        'rmse': rmse,
        'sharpe': sharpe,
        'total_return': total_return
    }
    config_path = save_dir / 'neuralprophet_config.pkl'
    joblib.dump(config, config_path)
    print(f'✓ Config saved: {config_path}')
    
    print('\\nTo load and use:')
    print('  model = torch.load(model_path / "model.pt")')
    print('  forecast = model.predict(new_data)')
else:
    print('⚠ No model to save')

## 8) Save Model

In [ ]:
if NeuralProphet is not None:
    fig, axes = plt.subplots(3, 2, figsize=(16, 12))
    
    # Plot 1: Forecast vs Actual (first 200 points)
    subset = forecast.iloc[:200]
    axes[0,0].plot(subset['ds'], subset['actual'], label='Actual', linewidth=2)
    axes[0,0].plot(subset['ds'], subset['forecast_1min'], label='1-min Forecast', alpha=0.7, linestyle='--')
    axes[0,0].fill_between(subset['ds'], subset['lower_bound'], subset['upper_bound'], alpha=0.2, label='80% Confidence')
    axes[0,0].set_title('Forecast vs Actual Price')
    axes[0,0].legend()
    axes[0,0].grid(True, alpha=0.3)
    axes[0,0].tick_params(axis='x', rotation=45)
    
    # Plot 2: Forecast errors
    axes[0,1].hist(forecast['error_1min'], bins=50, edgecolor='black', alpha=0.7)
    axes[0,1].set_title('Forecast Error Distribution')
    axes[0,1].set_xlabel('Error (Actual - Forecast)')
    axes[0,1].axvline(x=0, color='r', linestyle='--')
    axes[0,1].grid(True, alpha=0.3)
    
    # Plot 3: Equity curve
    axes[1,0].plot(equity)
    axes[1,0].set_title(f'Equity Curve (Trades: {len(trades)})')
    axes[1,0].set_xlabel('Time Step')
    axes[1,0].set_ylabel('Capital (₹)')
    axes[1,0].axhline(y=100000, color='r', linestyle='--', alpha=0.5, label='Initial')
    axes[1,0].legend()
    axes[1,0].grid(True, alpha=0.3)
    
    # Plot 4: Trade P&L distribution
    if trades:
        pnls = [t['pnl'] for t in trades]
        axes[1,1].hist(pnls, bins=30, edgecolor='black', alpha=0.7)
        axes[1,1].set_title('Trade P&L Distribution')
        axes[1,1].set_xlabel('P&L (₹)')
        axes[1,1].axvline(x=0, color='r', linestyle='--')
        axes[1,1].grid(True, alpha=0.3)
    
    # Plot 5: Signal distribution
    signal_counts = forecast['signal'].value_counts().sort_index()
    axes[2,0].bar(['SHORT', 'HOLD', 'LONG'], 
                  [signal_counts.get(i, 0) for i in range(3)])
    axes[2,0].set_title('Signal Distribution')
    axes[2,0].set_ylabel('Count')
    axes[2,0].grid(True, alpha=0.3, axis='y')
    
    # Plot 6: Confidence distribution
    axes[2,1].hist(forecast['confidence'], bins=30, edgecolor='black', alpha=0.7)
    axes[2,1].set_title('Forecast Confidence Distribution')
    axes[2,1].set_xlabel('Confidence')
    axes[2,1].set_ylabel('Frequency')
    axes[2,1].axvline(x=0.5, color='r', linestyle='--', alpha=0.5, label='Threshold')
    axes[2,1].legend()
    axes[2,1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
else:
    print('⚠ Skipping visualization')

## 7) Visualize Results

In [ ]:
if NeuralProphet is not None:
    def backtest_prophet_strategy(forecast_df, initial_capital=100000):
        """Backtest the forecast-based strategy"""
        capital = initial_capital
        position = 0  # 0=flat, 1=long, -1=short
        entry_price = 0
        equity = [capital]
        trades = []
        
        for i in range(1, len(forecast_df)):
            row = forecast_df.iloc[i]
            prev_row = forecast_df.iloc[i-1]
            
            signal = int(row['signal'])
            confidence = row['confidence']
            price = row['actual']
            
            # Only trade with sufficient confidence
            if confidence < 0.5:
                continue
            
            # Position sizing based on confidence
            pos_size = min(1.0, confidence * 1.2)
            
            # Calculate P&L if we have a position
            if position != 0:
                price_change = price - entry_price
                pnl = position * price_change * pos_size * 50  # 50 = lot size
                capital += pnl
                
                if abs(pnl) > 50:  # Track significant trades
                    trades.append({
                        'time': row['ds'],
                        'type': 'LONG' if position == 1 else 'SHORT',
                        'entry': entry_price,
                        'exit': price,
                        'pnl': pnl,
                        'confidence': confidence
                    })
            
            equity.append(capital)
            
            # Update position based on signal
            if signal == 2:  # LONG
                position = 1
                entry_price = price
            elif signal == 0:  # SHORT
                position = -1
                entry_price = price
            else:  # HOLD
                position = 0
        
        return equity, trades
    
    # Run backtest
    equity, trades = backtest_prophet_strategy(forecast)
    
    # Performance metrics
    total_return = (equity[-1] - 100000) / 100000 * 100
    returns = np.diff(equity) / equity[:-1]
    sharpe = returns.mean() / (returns.std() + 1e-10) * np.sqrt(252 * 60)
    max_dd = np.min(np.array(equity) - np.maximum.accumulate(equity))
    
    print('=== Backtest Results ===')
    print(f'Initial Capital: ₹100,000')
    print(f'Final Capital: ₹{equity[-1]:,.2f}')
    print(f'Total Return: {total_return:.2f}%')
    print(f'Sharpe Ratio: {sharpe:.3f}')
    print(f'Max Drawdown: ₹{max_dd:,.2f}')
    print(f'Number of Trades: {len(trades)}')
    if trades:
        avg_pnl = np.mean([t['pnl'] for t in trades])
        win_rate = sum(1 for t in trades if t['pnl'] > 0) / len(trades) * 100
        print(f'Avg P&L per trade: ₹{avg_pnl:.2f}')
        print(f'Win Rate: {win_rate:.1f}%')
else:
    print('⚠ Skipping backtest')

## 6) Backtest Strategy

In [ ]:
if NeuralProphet is not None:
    def generate_signals(forecast_df, lookback=30):
        """
        Generate entry/exit signals based on forecasts
        
        Strategy:
        - LONG if forecast shows upward movement > threshold with low uncertainty
        - SHORT if forecast shows downward movement > threshold with low uncertainty
        - HOLD otherwise (uncertain or flat forecast)
        """
        signals = []
        
        for i in range(len(forecast_df)):
            row = forecast_df.iloc[i]
            
            # Calculate expected price change over next 5 minutes
            current_price = row['actual']
            forecast_5min = row['forecast_5min']
            expected_change_pct = (forecast_5min - current_price) / current_price
            
            # Uncertainty factor (higher = less confident)
            uncertainty_pct = row['uncertainty'] / current_price
            
            # Thresholds
            move_threshold = 0.001  # 0.1% price move
            uncertainty_threshold = 0.003  # 0.3% uncertainty
            
            # Signal logic
            if uncertainty_pct > uncertainty_threshold:
                # Too uncertain - HOLD
                signal = 1
                reason = 'high_uncertainty'
            elif expected_change_pct > move_threshold:
                # Bullish forecast - LONG
                signal = 2
                reason = 'bullish_forecast'
            elif expected_change_pct < -move_threshold:
                # Bearish forecast - SHORT
                signal = 0
                reason = 'bearish_forecast'
            else:
                # Flat forecast - HOLD
                signal = 1
                reason = 'flat_forecast'
            
            signals.append({
                'signal': signal,
                'reason': reason,
                'expected_change_pct': expected_change_pct,
                'uncertainty_pct': uncertainty_pct,
                'confidence': 1 - min(1, uncertainty_pct / uncertainty_threshold)
            })
        
        return pd.DataFrame(signals)
    
    # Generate signals
    signals_df = generate_signals(forecast)
    forecast = pd.concat([forecast.reset_index(drop=True), signals_df], axis=1)
    
    # Signal distribution
    print('Signal Distribution:')
    print(forecast['signal'].value_counts().sort_index())
    print(f'\\nAverage confidence: {signals_df["confidence"].mean():.3f}')
    
    # Sample signals
    print('\\nSample signals:')
    print(forecast[['ds', 'actual', 'forecast_5min', 'signal', 'reason', 'confidence']].head(20))
else:
    print('⚠ Skipping signal generation')

## 5) Generate Entry/Exit Signals from Forecasts

In [ ]:
if NeuralProphet is not None:
    # Make predictions on test set
    forecast = model.predict(df_test)
    
    print(f'Forecast shape: {forecast.shape}')
    print(f'Columns: {forecast.columns.tolist()}')
    
    # Forecast contains:
    # - 'yhat1' to 'yhat10': predictions for next 10 steps
    # - 'yhat1 10%', 'yhat1 90%': uncertainty bounds
    # - 'ds': timestamp
    
    # Extract key forecasts
    forecast['actual'] = df_test['y'].values
    forecast['forecast_1min'] = forecast['yhat1']  # 1-minute ahead
    forecast['forecast_5min'] = forecast['yhat5']  # 5-minutes ahead
    forecast['forecast_10min'] = forecast['yhat10']  # 10-minutes ahead
    
    # Uncertainty bands
    forecast['lower_bound'] = forecast['yhat1 10.0%']
    forecast['upper_bound'] = forecast['yhat1 90.0%']
    forecast['uncertainty'] = forecast['upper_bound'] - forecast['lower_bound']
    
    print('\nSample forecasts:')
    print(forecast[['ds', 'actual', 'forecast_1min', 'forecast_5min', 'lower_bound', 'upper_bound']].head(10))
    
    # Calculate forecast errors
    forecast['error_1min'] = forecast['actual'] - forecast['forecast_1min']
    mae = np.abs(forecast['error_1min']).mean()
    rmse = np.sqrt((forecast['error_1min'] ** 2).mean())
    
    print(f'\n1-min Forecast Errors:')
    print(f'  MAE: {mae:.2f}')
    print(f'  RMSE: {rmse:.2f}')
else:
    print('⚠ Skipping forecast generation')

## 4) Generate Forecasts

In [ ]:
if NeuralProphet is None:
    print('⚠ Skipping training - NeuralProphet not available')
else:
    # Configure model for intraday trading
    model = NeuralProphet(
        growth='linear',              # Linear trend (for short intraday horizon)
        n_changepoints=10,            # Detect trend changes
        changepoints_range=0.9,       # Use 90% of data for changepoint detection
        trend_reg=1,                  # Regularize trend
        
        # Seasonality (daily patterns)
        yearly_seasonality=False,     # Not needed for intraday
        weekly_seasonality=False,     # Not needed for single day
        daily_seasonality=True,       # Important! Captures intraday patterns
        seasonality_mode='additive',
        seasonality_reg=0.1,
        
        # Neural network architecture
        n_forecasts=10,               # Forecast next 10 minutes
        n_lags=30,                    # Use last 30 minutes as features
        num_hidden_layers=2,          # 2 hidden layers
        d_hidden=32,                  # 32 units per layer
        
        # Training params
        learning_rate=0.01,
        epochs=50,                    # Quick training for demo
        batch_size=64,
        loss_func='Huber',            # Robust to outliers
        
        # Uncertainty quantification
        quantiles=[0.1, 0.9],         # 10th and 90th percentiles
    )
    
    # Add regressors (external features)
    model.add_lagged_regressor('volume', n_lags=5)
    model.add_lagged_regressor('high', n_lags=5)
    model.add_lagged_regressor('low', n_lags=5)
    
    print('Model configured. Training...')
    
    # Train
    metrics = model.fit(df_train, freq='1min', validation_df=df_test)
    
    print('\n✓ Training complete')
    print(f'Final MAE: {metrics["MAE"].iloc[-1]:.2f}')
    print(f'Final RMSE: {metrics["RMSE"].iloc[-1]:.2f}')

## 3) Configure & Train Neural Prophet Model

In [ ]:
processed_dir = repo_root / 'data' / 'processed_data'
df = None

if processed_dir.exists():
    for fn in ['train.csv', 'test.csv']:
        p = processed_dir / fn
        if p.exists():
            try:
                df = pd.read_csv(p, parse_dates=True, index_col=0)
                print(f'Loaded: {fn} - {len(df)} rows')
                break
            except Exception as e:
                print(f'Load error: {e}')

# Fallback: synthetic intraday data with realistic patterns
if df is None:
    print('Creating synthetic intraday time series...')
    np.random.seed(456)
    
    # Generate 5 days of 1-min intraday data
    dates = pd.date_range('2025-01-01 09:15', periods=5*375, freq='1min')
    
    # Base price with trend + intraday seasonality
    base = 22000
    trend = np.linspace(0, 500, len(dates))  # Upward trend
    
    # Intraday pattern (U-shaped: higher at open/close, lower midday)
    hour_of_day = dates.hour + dates.minute/60
    intraday_pattern = 50 * np.sin((hour_of_day - 9.25) * np.pi / 6) ** 2
    
    # Volatility clustering
    garch_vol = np.random.randn(len(dates)).cumsum() * 0.3
    
    # Combined price
    price = base + trend + intraday_pattern + garch_vol
    price += np.random.randn(len(dates)) * 5  # Noise
    
    df = pd.DataFrame({
        'open': price + np.random.randn(len(dates)) * 2,
        'high': price + np.abs(np.random.randn(len(dates)) * 4),
        'low': price - np.abs(np.random.randn(len(dates)) * 4),
        'close': price,
        'volume': np.random.randint(1000, 10000, len(dates))
    }, index=dates)
    print(f'Synthetic data: {len(df)} bars, {df.index[0]} to {df.index[-1]}')

# Neural Prophet requires specific format: 'ds' (datetime) and 'y' (target)
df_prophet = pd.DataFrame({
    'ds': df.index,
    'y': df['close'].values
})

# Optional: add regressors (volume, VWAP, etc.)
df_prophet['volume'] = df['volume'].values
df_prophet['high'] = df['high'].values
df_prophet['low'] = df['low'].values

print(f'Prophet format: {df_prophet.shape}')
print(df_prophet.head())

# Train/test split (80/20)
split_idx = int(0.8 * len(df_prophet))
df_train = df_prophet.iloc[:split_idx].copy()
df_test = df_prophet.iloc[split_idx:].copy()

print(f'\nTrain: {len(df_train)} samples')
print(f'Test: {len(df_test)} samples')

## 2) Load & Prepare Time Series Data

In [ ]:
# Install Neural Prophet (uncomment to install)
# !pip install neuralprophet pandas numpy matplotlib torch

import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

print('Importing NeuralProphet...')
try:
    from neuralprophet import NeuralProphet
    print('✓ NeuralProphet imported successfully')
except ImportError:
    print('⚠ NeuralProphet not installed. Run: pip install neuralprophet')
    NeuralProphet = None

# Torch/GPU check
try:
    import torch
    print(f'PyTorch version: {torch.__version__}')
    print(f'CUDA available: {torch.cuda.is_available()}')
    if torch.cuda.is_available():
        print(f'GPU: {torch.cuda.get_device_name(0)}')
except:
    print('PyTorch not available')

# Repo setup
repo_root = Path('/Users/muaazshaikh/SuperTrader.AI').resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
print(f'Repo: {repo_root}')

## 1) Install & Setup

# Intraday Entry/Exit Strategy - Neural Prophet
Time-series forecasting with Neural Prophet for price prediction + rule-based entry/exit.

**Neural Prophet advantages:**
- Built on PyTorch (fast, GPU-ready)
- Handles trends, seasonality, holidays automatically
- Interpretable forecasts with uncertainty intervals
- Good for multi-step ahead predictions

**Strategy:**
1. Forecast next N minutes of price movement
2. Generate entry/exit signals based on forecast
3. Use uncertainty bands for position sizing